Tarefa 1: Instale o ObsPy
Tarefa 2: Baixe este arquivo de exemplo (dados reais do SCEDC):
!wget https://examples.obspy.org/example.mseed

Tarefa 3: Escreva um script que:
Carregue o arquivo
Imprima as informações (stats) de cada trace
Faça um plot do sinal
Salve o plot como imagem

Perguntas:
 1. Qual a taxa de amostragem?
 2. Quantos canais existem?
 3. Qual a duração total do registro?

In [1]:
!pip install pandas numpy opspy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.1 MB/s eta 0:00:00a 0:00:01
  Using cached numpy-2.4.2-cp311-cp311-macosx_14_0_x86_64.whl.metadata (6.6 kB)
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11
ERROR: Could not find a version that satisfies the requirement opspy (from versions: none)
ERROR: No matching distribution found for opspy

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 9.9 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0


In [6]:
!pip install --upgrade --force-reinstall obspy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.0/17.0 MB 10.1 MB/s  0:00:01 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-2.4.2-cp311-cp311-macosx_14_0_x86_64.whl.metadata (6.6 kB)
  Using cached scipy-1.17.0-cp311-cp311-macosx_14_0_x86_64.whl.metadata (62 kB)
  Using cached matplotlib-3.10.8-cp311-cp311-macosx_10_12_x86_64.whl.metadata (52 kB)
  Using cached lxml-6.0.2-cp311-cp311-macosx_10_9_x86_64.whl.metadata (3.6 kB)
  Using cached setuptools-82.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached decorator-5.2.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached contourpy-1.3.3-cp311-cp311-macosx_10_9_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.61.1-cp311-cp311-macosx_10_9_x86_64.whl.metadata (114 kB)
  Using cached ki

In [8]:
import os 
os.chdir(r'/Users/alvarosamp/Documents/Projetos/TCC/Recaptulando/data')
home = os.getcwd()
print(home)

/Users/alvarosamp/Documents/Projetos/TCC/Recaptulando/data


In [12]:
import os
# switch to the project data directory (home variable set in previous cell)
os.chdir(home)
# ensure raw_data exists
os.makedirs('raw_data', exist_ok=True)
# download the example file using Python (works without wget)
import urllib.request, urllib.error
url = 'https://examples.obspy.org/example.mseed'
dest = os.path.join('raw_data', 'example.mseed')
try:
    urllib.request.urlretrieve(url, dest)
    print('Downloaded to', os.path.join(os.getcwd(), dest))
except urllib.error.HTTPError as e:
    print(f'HTTP error {e.code}: {e.reason} while downloading {url}')
    # try a curl fallback if available
    import shutil, subprocess
    if shutil.which('curl'):
        try:
            subprocess.check_call(['curl', '-L', '-o', dest, url])
            print('Downloaded with curl to', os.path.join(os.getcwd(), dest))
        except Exception as ex:
            print('curl failed:', ex)
    else:
        print('curl not available. To enable shell downloads install wget or curl (e.g. `brew install wget`).')

HTTP error 404: Not Found while downloading https://examples.obspy.org/example.mseed


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

Downloaded with curl to /Users/alvarosamp/Documents/Projetos/TCC/Recaptulando/data/raw_data/example.mseed


100   146  100   146    0     0    183      0 --:--:-- --:--:-- --:--:--   184


In [13]:
# Load the downloaded example file, print stats, plot each trace and save images
import os
from obspy import read
import matplotlib.pyplot as plt

file_path = os.path.join('raw_data', 'example.mseed')
print('Reading', file_path)
st = read(file_path)
# Number of channels/traces
n_channels = len(st)
print(f'Number of traces (channels): {n_channels}')
# Sampling rates (may differ between traces)
srs = sorted({tr.stats.sampling_rate for tr in st})
print('Sampling rate(s) (Hz):', srs)
# Total duration (from earliest start to latest end) in seconds
start_times = [tr.stats.starttime for tr in st]
end_times = [tr.stats.endtime for tr in st]
total_duration = max(end_times) - min(start_times)
print(f'Total duration (s): {total_duration}')
# Print stats for each trace and save a plot image
os.makedirs('raw_data/plots', exist_ok=True)
for i, tr in enumerate(st):
    print('--- Trace', i, '---')
    print(tr.stats)
    t = tr.times()  # time vector in seconds
    plt.figure(figsize=(10, 2.5))
    plt.plot(t, tr.data, color='k', linewidth=0.6)
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.title(f'{tr.id} — SR={tr.stats.sampling_rate} Hz')
    plt.tight_layout()
    outname = f'trace_{i}_{tr.id.replace('.', '_')}.png'
    outpath = os.path.join('raw_data', 'plots', outname)
    plt.savefig(outpath, dpi=150)
    plt.close()
    print('Saved plot to', outpath)

SyntaxError: f-string: unmatched '(' (2999035160.py, line 32)